# Lezione 2 · Regressione lineare applicata con scikit-learn

Ciao di nuovo! Nella lezione precedente hai visto cosa sia el aprendizaje automático o machine learning (ML) y come si organizza un proyecto de este tipo de principio a fin. Oggi parliamo di la tecnica più usata per iniziare con ML: la **regresión**.

Ti sei mai chiesto cómo una aplicación estima el precio de una vivienda, el consumo eléctrico de una fábrica o la demanda de un producto antes de que ocurra? Bene, oggi costruirai tu stesso, con le tue stesse mani, quel tipo di modello.

**Hai bisogno di Python installato insieme alle librerie NumPy, pandas, scikit-learn y Matplotlib.** Se lavori in locale con Anaconda o con pip, installale così (in Google Colab sono già preinstallate, quindi puoi saltare questa cella):

```bash
pip install numpy pandas scikit-learn matplotlib
```

Programmeremo ogni blocco insieme, passo dopo passo. Apri questo notebook en Google Colab o nel tuo ambiente locale ed esegui cada celda man mano che avanziamo.

## Cos'è esattamente la regressione?

Piensa en esto: quieres calcular el peso aproximado de una persona a partire da su altura. No tienes una báscula a mano, pero sabes que, en general, cuanto más alta es una persona, más pesa. Si reúnes datos de altura y peso de muchas personas, ¿podrías trazar una línea que resuma esa tendencia? Sí, y esa línea te permite, dada una altura nueva, calcular un peso probable.

Eso es justo lo que hace la regressione: busca la relación entre uno o varios datos que ya conoces (la altura) y un dato numérico que vuoi prevedere (el peso). Cuando esa relación se representa con una línea recta, hablas de **regresión lineal**.

¿Se te ocurren otros esempios similes? El precio de una vivienda a partire da su tamaño, el gasto energético de un edificio a partire da su superficie, o el prezzo di chiusura de una acción a partire da su comportamiento reciente. Hoy vamos a usar precisamente ese último esempio, el de un precio que varía día a día, porque es el que utiliza el material original de la certificación EITC/AI/MLP (European IT Certification / Artificial Intelligence / Machine Learning with Python) para introducir la regressione.

Una aclaración importante antes de seguir: todavía no vas a ver la fórmula matemática que hay detrás de la línea de regresión ni come si calcola su ajuste, eso llega en el capítulo 4 del manual. Hoy te centras en el uso práctico de la librería scikit-learn para entrenar, evaluar y usar un modelo de regresión.

## Features e labels: separare ciò che sai da ciò che vuoi predecir

In qualsiasi problema de regresión distingui due tipos de columnas. ¿Cuáles crees que son?

- Las **features** (caratteristiche o variabili predittive): i dati che già conosci e che usi come input
- L'**etichetta** (etiqueta o variabile obiettivo): il numero che vuoi prevedere

En el material original de esta lección, las features y el label si ottenevano da una serie di prezzi di borsa de Google scaricata in tempo reale con una API (interfaccia di programmazione de aplicaciones) financiera. Esa API non è più affidabile né gratuita al giorno d'oggi, así que en esta clase simuliamo una serie di prezzi con NumPy per poter praticare senza dipendere da servizi esterni. Il ragionamento y el código de scikit-learn sono identici a los del materiale originale: solo cambia l'origine dei dati.

Genera prima la serie simulata, con un seed fisso affinché il risultato sia sempre lo stesso ogni volta che esegui il notebook:

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_dias = 500
retorno_diario = np.random.normal(loc=0.0005, scale=0.02, size=n_dias)
precio_cierre = 100 * (1 + retorno_diario).cumprod()

fechas = pd.date_range(start="2022-01-03", periods=n_dias, freq="B")

df = pd.DataFrame({"Close": precio_cierre}, index=fechas)
df.index.name = "Fecha"

ruido_apertura = np.random.normal(loc=0, scale=0.005, size=n_dias)
df["Open"] = df["Close"].shift(1).fillna(df["Close"].iloc[0]) * (1 + ruido_apertura)

ruido_rango = np.random.uniform(low=0.002, high=0.02, size=n_dias)
df["High"] = df[["Open", "Close"]].max(axis=1) * (1 + ruido_rango)
df["Low"] = df[["Open", "Close"]].min(axis=1) * (1 - ruido_rango)

df["Volume"] = np.random.randint(1_000_000, 5_000_000, size=n_dias)

print(df.head())


Fíjate en lo che hai appena crear: un **passeggiata casuale** (random walk). Cada prezzo di chiusura es el anterior multiplicado por un pequeño retorno diario aleatorio. A partir de ese cierre simulado, calculas también un precio de apertura, un máximo, un mínimo y un volumen negociado, para imitar la forma de un dataset di borsa real.

Ahora bien, aquí viene algo importante: ¿crees que todas las columnas aportan lo mismo? La respuesta es no, y una regresión lineal simple no descubre por sí sola relaciones entre columnas. Si crees que la diferencia entre el máximo y el mínimo del día importa, tienes que calcularla tu stesso o tú misma y añadirla como una feature nueva. Vamos a crear dos columnas derivadas, tal como hace el material original:

In [ ]:
df["HL_percent"] = (df["High"] - df["Low"]) / df["Low"] * 100
df["PCT_change"] = (df["Close"] - df["Open"]) / df["Open"] * 100

columnas_features = ["Close", "HL_percent", "PCT_change", "Volume"]
print(df[columnas_features].head())


`HL_percent` recoge la volatilità del día (cuánto se movió el precio entre su máximo y su mínimo), y `PCT_change` recoge si el precio subió o bajó a lo largo de la sesión. Son esempios de cómo transformar columnas poco útiles por separado en features con significado.

Ahora, ¿qué falta? El label. Aquí vuoi prevedere el prezzo di chiusura, pero no el de hoy, sino el de unos días en el futuro. ¿Cómo lo consigues? Desplazando la columna `Close` hacia arriba un número fijo de filas con el metodo `shift` de pandas:

In [ ]:
FORECAST_COL = "Close"
FORECAST_OUT = 30  # número de días que queremos predecir hacia el futuro

df["label"] = df[FORECAST_COL].shift(-FORECAST_OUT)

print(df[["Close", "label"]].head())
print(df[["Close", "label"]].tail())


Fíjate en el resultado: en cada fila, `label` contiene el prezzo di chiusura real de 30 días después. Como no existen datos 30 días después de las últimas filas del dataset, esas filas quedan con valores nulos (`NaN`) en `label`. ¿Es esto un error? No, es exactamente lo que ocurre en el material original y es intencionado: esas filas sin label son las que más adelante usarás para predecir el futuro, porque son las únicas para las que todavía no conoces la respuesta.

## Addestrare e valutare el modelo con scikit-learn

Con las features y el label ya definidos, ¿qué toca ahora? Separar los datos en un conjunto de entrenamiento y un conjunto de prueba, entrenar el modelo con el primero y comprobar su rendimiento con el segundo.

¿Por qué es tan importante esta separación? Porque si evalúas el modelo con los mismos datos con los que lo entrenaste, no sabrás si de verdad aprendió un patrón general o si solo memorizó los datos de entrenamiento.

In [ ]:
from sklearn import preprocessing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = np.array(df[columnas_features])
X = preprocessing.scale(X)

X_lately = X[-FORECAST_OUT:]
X = X[:-FORECAST_OUT]

df.dropna(inplace=True)
y = np.array(df["label"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = LinearRegression(n_jobs=-1)
modelo.fit(X_train, y_train)

precision = modelo.score(X_test, y_test)
print("Precisión del modelo (R²):", round(precision, 4))


Rivediamo insieme cosa fa ogni riga, perché qui sta il nucleo di la clase de hoy:

- `preprocessing.scale(X)` ridimensiona ogni feature affinché tutte rimangano in un intervallo simile; così nessuna columna domina l'addestramento solo perché tener números más grandes
- `X_lately` salva le righe más recientes, quelle che non hanno etichetta perché corrispond ono al futuro che ancora desideri predecir
- `train_test_split` distribuisce il resto de filas en un 80% para entrenar y un 20% para probar, in modo casuale ma riproducibile grazie a `random_state`
- `LinearRegression()` crea el modelo; el parámetro `n_jobs=-1` gli indica di usare todos los core di il tuo processore disponibles para addestrare più velocemente
- `modelo.fit(X_train, y_train)` è l'addestramento: il modello aggiusta i suoi parametri interni affinché le sue previsioni si avvicinino il più possibile a `y_train`
- `modelo.score(X_test, y_test)` misura quanto bene prevede il modello su dati che non ha visto durante el entrenamiento

Cosa significa il numero che hai appena ottenuto? Ese último valor es el coefficiente di determinazione, conocido como R². Si muove tra 0 y 1: un R² de 1 significa che el modelo spiega perfettamente la variazione di label a partire da las features, mentre un R² vicino a 0 significa che el modelo non contribuisce molto più che indovinare il valore medio.

Con i dati che hai appena simular, è normale che ottenga un R² alto, perché l'etichetta è letteralmente el mismo prezzo di chiusura spostato in el tiempo y las features guardan una relación muy directa con él.

Un dato curioso del materiale originale: un modelo de regresión lineal simile raggiunge un R² de 0.96 (96%) sobre este tipo de datos, mentre un altro algoritmo, la regressione a vettori di supporto (SVR, por sus siglas en inglés), si ferma a 0.51 (51%) con los parametri predefiniti. Quale lezione pratica? Scikit-learn facilita il confronto algoritmos diversi con molto pochi cambi di codice, così provare più di uno prima di decidere è buona pratica.

Un'altra metrica usuale en regresión, sebbene non la calcoliamo oggi, es el errore quadratico medio (MSE, por sus siglas en inglés), che misura la media delle differenze al quadrato tra il valore previsto e quello reale. Quanto più basso, meglio. Vedrai l'MSE con più dettagli quando nel capitolo 4 del manual entri in come si calcola davvero una regresión lineal.

## Prevedere il futuro: forecasting

Ya tienes un modelo entrenado y evaluado. ¿Cuál es el paso final? Usarlo para predecir de verdad: aplicarlo a `X_lately`, ese bloque de filas recientes que dejaste fuera del entrenamiento porque no tenían label conocido.

In [ ]:
predicciones = modelo.predict(X_lately)

print("Predicciones para los próximos", FORECAST_OUT, "días:")
print(predicciones)


`modelo.predict()` recibe un conjunto de features y devuelve un valor numérico por cada fila: en este caso, el prezzo di chiusura estimado para cada uno de los 30 días siguientes al final de la serie simulata.

¿Cómo visualizamos esto junto al histórico? Añadimos una columna `Forecast` vacía y la vamos rellenando fila a fila con fechas futuras, tal como hace el material original:

In [ ]:
import datetime
import matplotlib.pyplot as plt

df["Forecast"] = np.nan

ultima_fecha = df.index[-1]
un_dia = datetime.timedelta(days=1)
siguiente_fecha = ultima_fecha + un_dia

for prediccion in predicciones:
    fila_vacia = [np.nan for _ in range(len(df.columns) - 1)]
    df.loc[siguiente_fecha] = fila_vacia + [prediccion]
    siguiente_fecha += un_dia

df["Close"].plot(label="Histórico simulado")
df["Forecast"].plot(label="Predicción a 30 días")
plt.legend(loc="lower right")
plt.xlabel("Fecha")
plt.ylabel("Precio de cierre simulado")
plt.title("Serie simulada y su predicción con regresión lineal")
plt.show()


El bucle recorre cada previsione y la añade al dataframe en una fila nueva, con una fecha posterior a la última que ya tenías. Como en este esempio simulado avanzamos un día natural en cada iteración, en vez de un día bursátil como haría un dataset real, alguna fecha puede caer en fin de semana; para el objetivo didáctico de hoy no supone ningún problema.

¿Por qué incluimos la fecha en el eje horizontal, si ni siquiera es una feature del modelo? Porque es lo que te permite leer el gráfico con sentido temporal: ver dónde termina lo conocido y dónde empieza lo predicho.

Con esto tienes el ciclo completo: simular datos, construir features y label, entrenar, evaluar y predecir. El capítulo 3 del manual retoma este mismo modelo para guardarlo en disco con pickling y hablar de escalado e interpretación con più dettagli, y el capítulo 4 te lleva a programar la regressione lineal desde cero.

Una advertencia honesta antes de cerrar: un modelo de regresión lineal como este es un buen punto de partida para aprender el flusso di lavoro, pero por sí solo no garantiza predicciones fiables en un contesto reale, y mucho menos en un mercato finanziario real. Cuantas más features relevantes incorpores y cuanto meglio entiendas los datos de partida, más útil será el modelo. La simplicidad de este esempio es intenzionale: te permite centrarte en el procedura antes de complicarlo con datos reales.

## Riassunto: lo que te llevas de esta clase

- La regresión predice un número continuo a partire da otras variables conocidas, como estimar un peso a partire da una altura
- Las features son las columnas de entrada y el label es la columna que vuoi prevedere
- Desplazar una columna con `shift` te permite convertir un valor presente en un label del futuro, con un horizonte de previsione que tú eliges
- `train_test_split` separa los datos en entrenamiento y prueba para poder medir el rendimiento del modelo su dati che no ha visto
- `LinearRegression().fit()` entrena el modelo, y `.score()` devuelve el R², un número entre 0 y 1 que indica cuánta variación del label explica el modelo
- `.predict()` aplica el modelo entrenado sobre datos nuevos, como el tramo final sin label que reservaste para el futuro

## Mettilo in pratica tú misma o tu stesso

Antes de pasar al notebook de práctica de esta clase, retoma el código de arriba y experimenta con dos cambios, uno detrás de otro, anotando el R² que obtienes en cada caso.

**¡Prueba esto! Cambia `FORECAST_OUT` de 30 a 5 y, después, a 60, y compara cómo varía la puntuación `.score()` del modelo en cada horizonte.**

**¡Anímate! Añade o quita una feature de `columnas_features` (por esempio, elimina `Volume` o añade una nueva columna calculada por ti) y vuelve a entrenar el modelo para ver si el R² meglioa o empeora.**

No hay una única respuesta correcta: el objetivo es que compruebes con tus propios ojos cómo el horizonte de previsione y la elección de features afectan a la calidad de un modelo de regresión.

Como sfida aggiuntiva, **prueba también a cambiar la seed di `np.random.seed()` por otro número** y vuelve a ejecutar todo el flujo. Verás que la serie simulata cambia por completo, pero el procedura (features, label, entrenamiento, evaluación y previsione) sigue siendo idéntico. Esa es precisamente la idea central de esta clase: el flusso di lavoro con scikit-learn es el mismo, tanto si trabajas con un precio bursátil real como con cualquier otra serie de datos numéricos que quieras predecir.

**¡Continúa ahora con il notebook de práctica de esta clase, donde aplicarás lo aprendido a un caso de uso nuevo!**

## Sigle usate en esta clase

- **ML**: Machine Learning o aprendizaje automático
- **EITC/AI/MLP**: European IT Certification / Artificial Intelligence / Machine Learning with Python
- **API**: Application Programming Interface o interfaccia di programmazione de aplicaciones
- **R²**: coefficiente di determinazione
- **SVR**: Support Vector Regression o regressione a vettori di supporto
- **MSE**: Mean Squared Error o errore quadratico medio